# Digit Count Classifier — Training Notebook

Trains a ResNet34-based binary classifier to predict whether a jersey crop shows **1 digit** (0–9) or **2 digits** (10–99).

**Requirements:** Colab GPU runtime (Runtime → Change runtime type → GPU)

In [ ]:
# Verify GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
# Mount Google Drive (for saving trained model later)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and checkout the MoviNet branch
%cd /content
!git clone https://github.com/magnimel/jersey-number-pipeline.git
%cd jersey-number-pipeline
!git checkout arjun/MoviNet

In [ ]:
# Install dependencies (do NOT install torch/torchvision — Colab already has CUDA versions)
!pip install -q SoccerNet gdown tqdm pillow

In [ ]:
# Download SoccerNet jersey-2023 data and extract
import os, zipfile, glob, shutil
from pathlib import Path

data_dir = Path("/content/jersey-number-pipeline/data")
soccer_dir = data_dir / "SoccerNet"

# Step 1: Download
from SoccerNet.Downloader import SoccerNetDownloader as SNdl
dl = SNdl(LocalDirectory=str(data_dir))
dl.downloadDataTask(task="jersey-2023", split=["train", "test", "challenge"])

# Step 2: Find all zips (SoccerNet downloader may nest them)
all_zips = glob.glob(str(data_dir / "**" / "*.zip"), recursive=True)
print(f"Found {len(all_zips)} zip files:")
for z in all_zips:
    print(f"  {z}")

# Step 3: Unzip everything in place
for zp in all_zips:
    parent = os.path.dirname(zp)
    print(f"Extracting {os.path.basename(zp)} into {parent}...")
    with zipfile.ZipFile(zp, 'r') as zf:
        zf.extractall(parent)

# Step 4: Find the actual jersey-2023 folder (wherever it ended up)
jersey_dirs = glob.glob(str(data_dir / "**" / "jersey-2023"), recursive=True)
if not jersey_dirs:
    # Maybe already at data/jersey-2023
    jersey_dirs = [str(data_dir / "jersey-2023")]

source = jersey_dirs[0]
print(f"\nFound data at: {source}")

# Step 5: Rename to SoccerNet
if os.path.exists(source) and str(source) != str(soccer_dir):
    if soccer_dir.exists():
        shutil.rmtree(soccer_dir)
    shutil.move(source, str(soccer_dir))
    print(f"Moved {source} -> {soccer_dir}")

# Step 6: Verify
checks = {
    "train_gt.json": soccer_dir / "train" / "train_gt.json",
    "test_gt.json": soccer_dir / "test" / "test_gt.json",
    "train/images": soccer_dir / "train" / "images",
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f"{'OK  ' if ok else 'MISS'} {name}: {path}")
    if not ok:
        all_ok = False

if all_ok:
    print("\nData ready!")
else:
    print("\nSome files missing — check the output above.")

In [ ]:
# Create val split (SoccerNet jersey-2023 only has train/test/challenge — no val)
import json, random, shutil
from pathlib import Path

seed = 42
val_ratio = 0.15
root = Path("/content/jersey-number-pipeline/data/SoccerNet")

train_gt_path = root / "train" / "train_gt.json"
train_img_dir = root / "train" / "images"
val_dir = root / "val"
val_img_dir = val_dir / "images"
val_gt_path = val_dir / "val_gt.json"

random.seed(seed)
val_img_dir.mkdir(parents=True, exist_ok=True)

with open(train_gt_path, "r") as f:
    gt = json.load(f)

tracklets = list(gt.keys())
random.shuffle(tracklets)
n_val = int(len(tracklets) * val_ratio)
val_ids = set(tracklets[:n_val])

val_gt = {}
new_train_gt = {}

for tid, label in gt.items():
    src = train_img_dir / tid
    if tid in val_ids:
        val_gt[tid] = label
        if src.exists():
            shutil.move(str(src), str(val_img_dir / tid))
    else:
        new_train_gt[tid] = label

with open(val_gt_path, "w") as f:
    json.dump(val_gt, f)
with open(train_gt_path, "w") as f:
    json.dump(new_train_gt, f)

print(f"train: {len(new_train_gt)} tracklets")
print(f"val:   {len(val_gt)} tracklets")

In [ ]:
# Train the digit count classifier (SGD, 15 epochs)
%cd /content/jersey-number-pipeline
!python3 digit_count_classifier.py --train --data ./data/SoccerNet

In [ ]:
# Test the trained model (prints accuracy, precision, recall, F1)
import glob, os
model_files = sorted(glob.glob('./experiments/digit_count_resnet34_*.pth'))
if not model_files:
    print("No trained model found in ./experiments/")
else:
    latest_model = model_files[-1]
    print(f"Testing with: {latest_model}")
    print(f"File size: {os.path.getsize(latest_model) / 1024 / 1024:.1f} MB")
    !python3 digit_count_classifier.py --data ./data/SoccerNet --trained_model_path {latest_model}

In [ ]:
# Save trained model to Google Drive
import shutil, os, glob
from pathlib import Path

model_files = sorted(glob.glob('./experiments/digit_count_resnet34_*.pth'))
if not model_files:
    print("No model to save!")
else:
    latest_model = model_files[-1]
    drive_dest = Path('/content/drive/MyDrive/jersey-number-pipeline-models/')
    drive_dest.mkdir(parents=True, exist_ok=True)
    dest_path = drive_dest / os.path.basename(latest_model)
    shutil.copy(latest_model, str(dest_path))
    print(f"Model saved to: {dest_path}")
    print(f"Size: {os.path.getsize(str(dest_path)) / 1024 / 1024:.1f} MB")

    # Verify it loads correctly
    import torch
    from networks import DigitCountClassifier
    model = DigitCountClassifier()
    state = torch.load(str(dest_path), map_location='cpu', weights_only=True)
    model.load_state_dict(state)
    print("Load verification: OK")